# 01 章 / 第 3 节 · 手写一个对齐 Qwen 架构的 decoder-only

## 学习目标
- 手写 RMSNorm / RoPE / SwiGLU / GQA / KV cache 五个核心组件
- 拼成一个 ~25M 参数的 decoder-only,结构对齐 Qwen2.5-mini
- 做 forward / generate 单元自测,验证形状和因果性都正确

## 对应八股
`liguodongiot/llm-action`:**`llm-interview/llm-algo.md`**(LLM 架构章节)


## 1. 概念背景

现代 decoder-only LLM(Qwen / Llama / Mistral)架构高度收敛,核心组件就五件:

| 组件 | 替代了什么 | 为什么 |
|---|---|---|
| **RMSNorm** | LayerNorm | 去掉 mean 中心化,算得快、效果差不多 |
| **RoPE** | 位置 embedding | 旋转矩阵编码相对位置,长度外推性好 |
| **SwiGLU** | ReLU / GELU MLP | gated 激活,实验表现更好 |
| **GQA** | MHA(多头注意力) | KV 头数 < Q 头数,KV cache 显存掉一档 |
| **KV cache** | 重算历史 KV | decode 阶段 O(1) 拼接,不是 O(n) 重算 |

本节把这五件**全部手写一遍**,看完心里就有数 —— 后面读 transformers 源码、读 vLLM 源码、面试聊架构,都从这里出发。


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()


## 3. 核心实现


### 3.1 RMSNorm —— 比 LayerNorm 更轻

公式:`y = x / sqrt(mean(x²) + eps) * gamma`。注意**不减均值**。


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D) -> compute over last dim
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


# 自测:输出形状不变,均方根 ≈ 1
_x = torch.randn(2, 4, 16)
_y = RMSNorm(16)(_x)
print("shape:", _y.shape, "  rms ≈ 1:", _y.pow(2).mean(-1).sqrt().mean().item())


### 3.2 RoPE —— 旋转位置编码

把每对相邻维度看作复平面上一个向量,按位置 `m` 旋转角度 `m * theta`。Q 和 K 都旋转后,
点积自然带上**相对位置**信息(因为旋转矩阵满足 `R(m)·R(n)ᵀ = R(m-n)`)。

实现细节:预计算 cos/sin 表,forward 里查表 → reshape 成对 → 旋转 → 拼回。


In [ ]:
def precompute_rope_cache(head_dim: int, max_seq_len: int, base: float = 10000.0,
                          device=None, dtype=torch.float32):
    """返回 cos, sin 两个形状 (max_seq_len, head_dim // 2) 的张量。"""
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(t, inv_freq)  # (T, head_dim/2)
    return freqs.cos().to(dtype), freqs.sin().to(dtype)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """x: (B, n_head, T, head_dim);返回同形张量。"""
    # 把最后一维拆成 [偶数索引, 奇数索引] 两半
    x1, x2 = x[..., 0::2], x[..., 1::2]
    # cos/sin: (T, head_dim/2) -> 广播到 (1, 1, T, head_dim/2)
    c = cos[None, None, : x.shape[-2], :]
    s = sin[None, None, : x.shape[-2], :]
    # 旋转: [x1 cos - x2 sin, x1 sin + x2 cos] -> interleave 回去
    y1 = x1 * c - x2 * s
    y2 = x1 * s + x2 * c
    return torch.stack([y1, y2], dim=-1).flatten(-2)


# 自测:RoPE 后的 q·k 在相对位置相同时应该一致
_h, _d = 4, 16
_q = torch.randn(1, _h, 8, _d)
_k = torch.randn(1, _h, 8, _d)
_c, _s = precompute_rope_cache(_d, 8)
_q_r = apply_rope(_q, _c, _s)
_k_r = apply_rope(_k, _c, _s)
print("RoPE shape OK:", _q_r.shape)


### 3.3 SwiGLU MLP —— gated 激活的 feedforward

公式:`SwiGLU(x) = silu(W₁x) * (W₂x)`,再投回 d_model。`silu(x) = x * sigmoid(x)`(也叫 swish)。


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        # 一次算两路(gate / up),效率更好
        self.gate_up = nn.Linear(dim, hidden_dim * 2, bias=False)
        self.down = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


print("SwiGLU OK:", SwiGLU(16, 64)(torch.randn(2, 4, 16)).shape)


### 3.4 GQA Attention —— KV 头数 < Q 头数

`MHA → MQA → GQA` 一脉:
- **MHA**:每个 Q 头有自己的 K/V 头,显存大、效果最好
- **MQA**:所有 Q 头共享一对 K/V,显存最小、效果掉
- **GQA**:分组,组内 Q 头共享一对 K/V —— 显存折中、效果几乎不掉(Llama / Qwen 默认)

实现要点:Q 头数 = `n_head`,KV 头数 = `n_kv_head`,attention 计算前把 KV 头数 `repeat_interleave` 到 Q 头数。


In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """x: (B, n_kv_head, T, head_dim) -> (B, n_kv_head * n_rep, T, head_dim)"""
    if n_rep == 1:
        return x
    return x.repeat_interleave(n_rep, dim=1)


class GQA(nn.Module):
    def __init__(self, dim: int, n_head: int, n_kv_head: int, max_seq_len: int = 2048):
        super().__init__()
        assert dim % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = dim // n_head

        self.q_proj = nn.Linear(dim, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        cos, sin = precompute_rope_cache(self.head_dim, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor, kv_cache: dict | None = None) -> torch.Tensor:
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        # 应用 RoPE 到 q / k(注意起始位置:有 cache 时从 cache 长度开始)
        start = kv_cache["len"] if kv_cache else 0
        cos = self.rope_cos[start : start + T]
        sin = self.rope_sin[start : start + T]
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # KV cache:把历史 k / v 拼到前面
        if kv_cache is not None:
            if kv_cache["len"] > 0:
                k = torch.cat([kv_cache["k"], k], dim=2)
                v = torch.cat([kv_cache["v"], v], dim=2)
            kv_cache["k"] = k
            kv_cache["v"] = v
            kv_cache["len"] = k.shape[2]

        # GQA:把 K/V 头复制到 Q 头数
        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        # 用 PyTorch 内建 SDPA(自动 flash / mem-efficient),is_causal 自动加上因果 mask
        is_causal = kv_cache is None or kv_cache["len"] == T  # 只在不带 cache 或首次 prefill 时加因果 mask
        out = F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.o_proj(out)


# 自测
_attn = GQA(dim=128, n_head=8, n_kv_head=2)
print("GQA OK:", _attn(torch.randn(2, 16, 128)).shape)


### 3.5 Transformer Block 拼装

标准 pre-norm 结构:`x + attn(norm(x)) → x + mlp(norm(x))`。


In [ ]:
class Block(nn.Module):
    def __init__(self, dim: int, n_head: int, n_kv_head: int, mlp_hidden: int, max_seq_len: int):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GQA(dim, n_head, n_kv_head, max_seq_len)
        self.mlp_norm = RMSNorm(dim)
        self.mlp = SwiGLU(dim, mlp_hidden)

    def forward(self, x: torch.Tensor, kv_cache: dict | None = None) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x), kv_cache=kv_cache)
        x = x + self.mlp(self.mlp_norm(x))
        return x


### 3.6 NanoDecoder 整体模型


In [ ]:
from dataclasses import dataclass


@dataclass
class NanoConfig:
    vocab_size: int = 8192
    dim: int = 256
    n_layer: int = 4
    n_head: int = 8
    n_kv_head: int = 2
    mlp_hidden: int = 768
    max_seq_len: int = 512
    tied_embedding: bool = True  # 输入 embed 与输出 head 共享权重(省参数)


class NanoDecoder(nn.Module):
    def __init__(self, cfg: NanoConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.dim)
        nn.init.normal_(self.embed.weight, std=0.02)  # GPT 默认 init,让初始 logits 范围正常
        self.blocks = nn.ModuleList([
            Block(cfg.dim, cfg.n_head, cfg.n_kv_head, cfg.mlp_hidden, cfg.max_seq_len)
            for _ in range(cfg.n_layer)
        ])
        self.final_norm = RMSNorm(cfg.dim)
        if cfg.tied_embedding:
            self.lm_head = lambda x: x @ self.embed.weight.t()
        else:
            self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, kv_caches: list[dict] | None = None) -> torch.Tensor:
        x = self.embed(idx)
        for i, blk in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches is not None else None
            x = blk(x, kv_cache=cache)
        x = self.final_norm(x)
        return self.lm_head(x)  # (B, T, vocab)

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int = 32,
                 temperature: float = 1.0, top_k: int | None = None) -> torch.Tensor:
        """用 KV cache 的 generate。"""
        self.eval()
        kv_caches = [{"k": None, "v": None, "len": 0} for _ in range(self.cfg.n_layer)]
        # 首次 prefill
        logits = self(idx, kv_caches=kv_caches)
        for _ in range(max_new_tokens):
            next_logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(next_logits, top_k)
                next_logits[next_logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_id], dim=1)
            logits = self(next_id, kv_caches=kv_caches)
        return idx


## 4. 跑通 + 可视化


In [ ]:
cfg = NanoConfig()
model = NanoDecoder(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {n_params/1e6:.2f} M")
print(f"配置: dim={cfg.dim}, n_layer={cfg.n_layer}, n_head={cfg.n_head}, n_kv_head={cfg.n_kv_head}")

# Forward 形状测试
x = torch.randint(0, cfg.vocab_size, (2, 32))
logits = model(x)
print(f"\nforward: input {x.shape} -> logits {logits.shape}")
assert logits.shape == (2, 32, cfg.vocab_size)
print("[OK] 形状正确")


In [ ]:
# Generate sanity check(随机权重,只验证机制能跑通)
prompt = torch.randint(0, cfg.vocab_size, (1, 4))
gen = model.generate(prompt, max_new_tokens=16, top_k=50)
print(f"prompt 长度: {prompt.shape[1]}")
print(f"生成后长度: {gen.shape[1]} (期望 {4 + 16})")
assert gen.shape == (1, 4 + 16)
print("[OK] generate 工作正常")


In [ ]:
# 因果性验证:改变末位 token 不应该影响前面位置的 logits
x1 = torch.randint(0, cfg.vocab_size, (1, 16))
x2 = x1.clone()
x2[0, -1] = (x1[0, -1] + 1) % cfg.vocab_size

with torch.no_grad():
    l1 = model(x1)
    l2 = model(x2)

diff_front = (l1[:, :-1] - l2[:, :-1]).abs().max().item()
diff_back  = (l1[:, -1:] - l2[:, -1:]).abs().max().item()
print(f"前 15 位 logits 差异(应为 0): {diff_front:.2e}")
print(f"末 1 位 logits 差异(应 > 0): {diff_back:.2e}")
assert diff_front < 1e-5
print("[OK] 因果 mask 正确")


## 5. 踩坑记录

- **RoPE 起始位置**:带 KV cache 的 decode 阶段,新 token 的 RoPE 角度要从 `cache_len` 开始,不是从 0。我们用 `start = kv_cache["len"]` 处理。
- **GQA `repeat_interleave` vs `repeat`**:`repeat_interleave(n_rep, dim=1)` 把 `[a, b]` 复制成 `[a, a, b, b]`,这才是 GQA 想要的(同组 Q 头共享同一个 KV);`repeat` 会变成 `[a, b, a, b]`,语义错。
- **`F.scaled_dot_product_attention` 的 `is_causal`**:只在不带 cache 或首次 prefill 时启用;decode 阶段每次只送 1 个新 token,所有历史 K/V 都该看,不能再加因果 mask。
- **tied embedding 节省参数**:`vocab × dim` 是 nano 模型最大的一块参数(8192×256 = 2.1M),tie 起来直接砍一半。


## 6. 延伸阅读

- [[Slipbox/rope-rotary-embedding]] / [[Slipbox/rmsnorm-vs-layernorm]] / [[Slipbox/gqa-mqa-mha]]
- 论文:`/paper fetch 2104.09864` (RoFormer / RoPE)
- 论文:`/paper fetch 2305.13245` (GQA)
- 对比阅读:Qwen2.5 官方模型代码 [`transformers/models/qwen2`](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen2/modeling_qwen2.py),架构完全一致,只是工程化更复杂
